# 4. Simple Hero Encoding

## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb

# Local utilities
from lib.data_management import ( load_match_tables_jsonl,
    load_pro_matches_parquet
)
from lib.experiment_runner import run_experiment

from dota_oracle_common.postgresql import DatabaseManager

from dota_oracle_pipeline.feature_engineering import HeroesFeatureCreator


pd.set_option("display.max_columns", 100)
pd.set_option('display.max_rows', 50)





sns.set_theme(style="whitegrid")


In [2]:
os.environ["ENABLE_LOKI_LOGGING"] = "false"

# Create a database session factory for this notebook session
local_session = DatabaseManager.get_session_factory()

2025-09-30 06:15:01,436 - dota_oracle_common.postgresql - INFO - Creating database engine with pool_size=10 and max_overflow=5
2025-09-30 06:15:01,460 - dota_oracle_common.postgresql - INFO - Successfully initialized database for 'dota2' at localhost


In [3]:
Models = {
    'Logistic Regression': LogisticRegression(
        random_state=42,
        max_iter=1000  # Increase if convergence issues
    ),
    
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1  # Use all cores
    ),
    
    'XGBoost': xgb.XGBClassifier(
        random_state=42,
        eval_metric='logloss',  # Suppress warning
        n_estimators=100
    ),
    
    'LightGBM': lgb.LGBMClassifier(
        random_state=42,
        objective='binary',
        verbosity=-1,  # Suppress output
        n_estimators=100
    )
}

## 2. Load Dataset


In [4]:
df = load_pro_matches_parquet("../data/pro_match_dataset_.parquet")
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40525 entries, 0 to 40524
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype              
---  ------       --------------  -----              
 0   match_id     40525 non-null  int64              
 1   patch        40525 non-null  object             
 2   radiant_win  40525 non-null  bool               
 3   start_time   40525 non-null  datetime64[ns, UTC]
dtypes: bool(1), datetime64[ns, UTC](1), int64(1), object(1)
memory usage: 989.5+ KB


In [5]:
all_match_tables = load_match_tables_jsonl("../data/pro_match_tables.jsonl")
len(all_match_tables)

40542

In [6]:
# Ensure all_match_data are sorted chronologically by start_time
df.sort_values(by="start_time", inplace=True)

# Split data into training and testing sets, most recent 20% of data as test set
train_percentage = 0.8
n_train = int(len(df) * train_percentage)


train_df, test_df = df[:n_train], df[n_train:]

In [7]:
train_match_ids = train_df['match_id'].unique()
test_match_ids = test_df['match_id'].unique()

len(train_match_ids), len(test_match_ids)

(32420, 8105)

In [8]:
train_outcome_df = train_df[['match_id', 'radiant_win']]
test_outcome_df = test_df[['match_id', 'radiant_win']]

In [9]:
raw_hero_features = HeroesFeatureCreator().create_hero_features(all_match_tables)
raw_hero_df = pd.DataFrame([feat.model_dump() for feat in raw_hero_features])

raw_hero_df.head(5)

2025-09-30 06:15:24,435 - dota_oracle_pipeline.feature_engineering.heroes_features_creator - INFO - Created 40542 hero features


,match_id,hero_picks
0,7875299320,"[55, 18, 72, 74, 68, 69, 44, 96, 7, 121]"
1,7875309074,"[11, 104, 64, 67, 33, 16, 19, 63, 18, 129]"
2,7875316743,"[71, 69, 1, 87, 11, 10, 106, 26, 120, 103]"
3,7875322386,"[80, 91, 48, 38, 40, 72, 88, 120, 136, 95]"
4,7875314713,"[10, 5, 74, 119, 49, 128, 18, 98, 106, 111]"


In [10]:
hero_with_outcome_df_train = raw_hero_df.merge(df[['match_id', 'radiant_win']], on='match_id', how='inner')
hero_with_outcome_df_test = raw_hero_df.merge(df[['match_id', 'radiant_win']], on='match_id', how='inner')

## 4.1 MultiLabelbinarser Encoding 

In [11]:
from dota_oracle_pipeline.feature_transformation.feature_encoder import FeatureEncoder
from dota_oracle_common.repositories.heroes_repository import HeroesRepository

async with local_session() as session:
    heros_repo = HeroesRepository(session)
    hero_map = await heros_repo.get_hero_id_map()

In [12]:
feature_encoder = FeatureEncoder(hero_map=hero_map)

encoded_hero_features_train = feature_encoder.transform_batch(hero_with_outcome_df_train)
encoded_hero_features_test = feature_encoder.transform_batch(hero_with_outcome_df_test)

In [19]:
encoded_hero_features_train.shape, encoded_hero_features_test.shape

((40525, 127), (40525, 127))

In [20]:
run_experiment(
    feature_sets_train=[
        encoded_hero_features_train,
    ],
    feature_sets_test=[
        encoded_hero_features_test,
    ],
    y_test_df=test_outcome_df,
    y_train_df=train_outcome_df,
    models_dict=Models,
)

Starting new experiment run...


Training Logistic Regression...
Logistic Regression Accuracy: 0.519 (51.9%)

Training Random Forest...
Random Forest Accuracy: 0.494 (49.4%)

Training XGBoost...


/home/ubuntu/projects/dota2pred/.venv/lib/python3.11/site-packages/xgboost/data.py:399: UserWarning: Sparse arrays from pandas are converted into dense.
  warnings.warn("Sparse arrays from pandas are converted into dense.")
/home/ubuntu/projects/dota2pred/.venv/lib/python3.11/site-packages/xgboost/data.py:399: UserWarning: Sparse arrays from pandas are converted into dense.
  warnings.warn("Sparse arrays from pandas are converted into dense.")
/home/ubuntu/projects/dota2pred/.venv/lib/python3.11/site-packages/xgboost/data.py:399: UserWarning: Sparse arrays from pandas are converted into dense.
  warnings.warn("Sparse arrays from pandas are converted into dense.")


XGBoost Accuracy: 0.504 (50.4%)

Training LightGBM...
LightGBM Accuracy: 0.512 (51.2%)

Experiment completed. Leaderboard:
                  name  accuracy
0  Logistic Regression  0.519432
1             LightGBM  0.512400
2              XGBoost  0.503640
3        Random Forest  0.494139


### Experiment: Evaluating a Simple Multi-Hot Draft Encoding

#### Hypothesis and Method

As a baseline approach to feature representation, an experiment was conducted to see if the entire 10-hero draft could be represented as a single, simple feature vector. The `MultiLabelBinarizer` from scikit-learn was used for this purpose.

This technique transforms the list of 10 hero IDs for each match into a wide, sparse "multi-hot" vector. The resulting feature set has a column for every hero in the game, where a `1` indicates the hero was picked in that match (by either team) and a `0` indicates they were not.

#### Results

The predictive performance of this feature set was evaluated across all models:

| Model Name | `MultiLabelBinarizer` Accuracy |
| :--- | :--- |
| Logistic Regression | 51.94% |
| LightGBM | 51.24% |
| XGBoost | 50.36% |
| Random Forest | 49.41% |

The results showed that the best model, Logistic regression performed worse than the first baseline model of predicting all radiant win, which basically has almost no significant predictive signal. 

#### Analysis: The Flaw of a Team-Agnostic Representation

The complete failure of this approach provides a critical insight into the nature of the prediction problem. The `MultiLabelBinarizer` treats the 10-hero draft as a single "bag of heroes," completely erasing the fundamental distinction between the Radiant and Dire teams.

For example, if two heros, `Disruptor` and `Naga Siren` are picked in two matches, the former had both on the same team, and in the other opposite teams, the model treats them as the same. Domain knowledge tells us that there is a synergistic effect of these two heros being picked together. 

Since the prediction target is `radiant_win`, providing the model with features that are **symmetrical** with respect to the teams makes the learning task extremely difficult. 

#### Moving forward

We will need to engineer features which explicitly preserve and represent the distinct compositions of the Radiant and Dire teams separately.

## Word to Vector Embedding 

The next experiment will explore word embedding techniques using Gensim's powerful `Word2Vec` implementation. This approach allows us to transform a team's hero draft from a simple list of IDs into a dense, meaningful numerical vector that captures complex relationships between heroes.

The core idea is straightforward. We treat each team's draft as a "sentence" and each hero as a "word." For example, a team composition of Disruptor, Naga Siren, Centaur, Luna, and Rubick becomes the sentence `['disruptor', 'naga_siren', 'centaur', 'luna', 'rubick']`. By training a `Word2Vec` model on thousands of these "draft sentences," the model learns the co-occurrence patterns between heroes. Heroes that are frequently picked together (synergies) or on the same team will have their vectors pushed closer together in a high-dimensional space.

To make this model predictive, we introduce a simple but powerful tweak: **contrastive learning**. We prepend the match outcome (`'WIN'` or `'LOSS'`) to each draft sentence. Now, a sentence looks like `['WIN', 'disruptor', 'naga_siren', ...]`. This forces the `Word2Vec` algorithm to learn not just which heroes appear together, but which groups of heroes are associated with winning versus losing. The model will implicitly learn vector representations where "winning" heroes are oriented in one direction of the vector space, and "losing" heroes in another.

Finally, once the model is trained and we have a unique vector for each hero, we can generate features for any given match. We will calculate the **team centroid**—the average of the five hero vectors for each team. This single vector represents the overall strategic profile of a team's draft. From these centroids, we can derive powerful predictive features, such as the vector difference between the two teams and their cosine similarity, to quantify the matchup's advantage.

In [13]:
from lib.hero_features.word_to_vec import Word2VecFeatureCreator

w2v_creator = Word2VecFeatureCreator(vector_size=32)
w2v_creator.fit(hero_with_outcome_df_train)

Step 1/3: Creating contrastive training sentences...
Step 2/3: Training contrastive Word2Vec model...
Step 3/3: Extracting final hero embedding map...
Fit complete. Embeddings for 126 heroes learned.


In [14]:
print("\n--- Transforming the training data ---")
w2v_features_train = w2v_creator.transform(hero_with_outcome_df_train)

# Transform the test data
print("\n--- Transforming the test data ---")
w2v_features_test = w2v_creator.transform(hero_with_outcome_df_test)


--- Transforming the training data ---
Calculating features using vector concatenation...
Transformation complete. Final shape: (40525, 321)

--- Transforming the test data ---
Calculating features using vector concatenation...
Transformation complete. Final shape: (40525, 321)


In [15]:
w2v_features_train.shape, w2v_features_test.shape

((40525, 321), (40525, 321))

In [16]:
w2v_features_train.shape

(40525, 321)

In [17]:
run_experiment(
    feature_sets_train=[
        w2v_features_train,
    ],
    feature_sets_test=[
        w2v_features_test,
    ],
    y_test_df=test_outcome_df,
    y_train_df=train_outcome_df,
    models_dict=Models,
)

Starting new experiment run...


Training Logistic Regression...
Logistic Regression Accuracy: 0.533 (53.3%)

Training Random Forest...
Random Forest Accuracy: 0.522 (52.2%)

Training XGBoost...
XGBoost Accuracy: 0.520 (52.0%)

Training LightGBM...
LightGBM Accuracy: 0.517 (51.7%)

Experiment completed. Leaderboard:
                  name  accuracy
0  Logistic Regression  0.532634
1        Random Forest  0.521777
2              XGBoost  0.519679
3             LightGBM  0.516841
